# Provider Daily Economics

This notebook uses the research mart to analyze daily provider token volume and estimated revenue.

Revenue here is an estimate based on observed token activity plus as-of OpenRouter catalog pricing snapshots. It is not audited provider revenue.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

BASE_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print(f"Base dir: {BASE_DIR}")


In [ ]:
from research_data.api import build_daily_provider_economics, provider_revenue_daily, provider_tokens_daily

provider_economics = build_daily_provider_economics(base_dir=BASE_DIR, refresh=False)
print("Rows:", len(provider_economics))
provider_economics.head()


## Selected Providers


In [ ]:
selected_providers = ["openai", "anthropic", "google"]
tokens_daily = provider_tokens_daily(selected_providers, base_dir=BASE_DIR, refresh=False)
revenue_daily = provider_revenue_daily(selected_providers, base_dir=BASE_DIR, refresh=False)

print(tokens_daily.tail(10).to_string(index=False))
print(revenue_daily.tail(10).to_string(index=False))


In [ ]:
provider_summary = (
    revenue_daily.groupby("provider_slug", as_index=False)
    .agg(
        total_tokens=("total_tokens", "sum"),
        estimated_revenue=("estimated_revenue", "sum"),
        priced_rows=("pricing_join_status", lambda s: int((s.str.startswith("matched")).sum())),
        total_rows=("pricing_join_status", "count"),
    )
    .sort_values("estimated_revenue", ascending=False)
)
provider_summary


In [ ]:
daily_rollup = (
    revenue_daily.groupby(["usage_date", "provider_slug"], as_index=False)
    .agg(total_tokens=("total_tokens", "sum"), estimated_revenue=("estimated_revenue", "sum"))
    .sort_values(["usage_date", "provider_slug"])
)
daily_rollup.tail(15)
